# 🐍 Learn Python for Data Engineering
## Through Real Example: `validate_dbs.py`

This notebook breaks down a database validation script line by line.

**What this script does:** Connects to MySQL and PostgreSQL databases to verify they're working.

**Why it matters for Data Engineers:** Database connectivity is fundamental - you'll do this daily!

---
## 📦 Part 1: Imports (Loading External Tools)

Python doesn't have everything built-in. We use `import` to load external **modules/libraries**.

Think of it like: "I need a screwdriver, let me grab it from the toolbox."

In [ ]:
# SYNTAX: import <module_name>
# This loads the entire module so you can use its functions

import mysql.connector  # Library to connect to MySQL databases
import psycopg2         # Library to connect to PostgreSQL databases  
import sys              # Built-in module for system operations (like exit codes)

# WHY these libraries?
# - mysql.connector: Official MySQL driver for Python
# - psycopg2: Most popular PostgreSQL adapter for Python
# - sys: Lets us exit the script with status codes (0=success, 1=failure)

# DATA ENGINEER TIP: 
# You'll use these two database libraries A LOT. Memorize them!

### 💡 Import Variations You'll See
```python
import pandas as pd           # Import with alias (shortcut name)
from datetime import datetime # Import specific item from module
import os                     # Import entire module
```

---
## 🔧 Part 2: Functions (Reusable Code Blocks)

Functions let you write code once and reuse it. This is **DRY** principle: Don't Repeat Yourself.

### Syntax:
```python
def function_name(parameter1, parameter2):
    """Docstring: describes what function does"""
    # code here
    return result
```

### 2.1 The Main Function: `test_connection`

This is a **generic** function that can test ANY database connection.

Let's break it down piece by piece:

In [ ]:
# FUNCTION DEFINITION
# def = "define a function"
# **kwargs = "keyword arguments" - accepts any named parameters as a dictionary

def test_connection(db_type, connector_func, **kwargs):
    """
    Test a database connection.
    
    Parameters:
    - db_type: str, like 'MySQL' or 'PostgreSQL' (for display)
    - connector_func: function, the actual connect function to call
    - **kwargs: dict, connection parameters (host, user, password, database)
    
    Returns:
    - bool: True if connected, False if failed
    """
    
    # f-string: f"text {variable}" - embeds variables in strings
    # This is the MODERN way to format strings in Python 3.6+
    print(f"Testing {db_type} Connection...")
    
    # LIST: ordered collection, created with []
    # Here we create a list of hosts to try
    hosts_to_try = [kwargs['host']]  # kwargs is a dict, access with ['key']
    
    # CONDITIONAL: if <condition>:
    # 'not in' checks if value is NOT in a list
    if kwargs['host'] not in ['localhost', '127.0.0.1']:
        hosts_to_try.append('localhost')  # .append() adds item to end of list
    
    # FOR LOOP: iterate over each item in a collection
    for host in hosts_to_try:
        # TRY-EXCEPT: Error handling (CRITICAL for data engineering!)
        # "Try this code, if it fails, do something else"
        try:
            print(f"  Attempting connection to host: {host}...")
            
            # .copy() creates a new dict (so we don't modify original)
            conn_kwargs = kwargs.copy()
            conn_kwargs['host'] = host  # Update the host value
            
            # CALLING A FUNCTION STORED IN A VARIABLE
            # connector_func is either mysql.connector.connect or psycopg2.connect
            # **conn_kwargs "unpacks" the dict into named parameters
            # Same as: connect(host='mysql', user='devuser', password='devpassword', database='devdb')
            conn = connector_func(**conn_kwargs)
            
            # CURSOR: Think of it as a "pointer" to execute SQL and fetch results
            cursor = conn.cursor()
            
            # CONDITIONAL with different SQL for different databases
            if db_type == 'MySQL':
                cursor.execute("SELECT VERSION()")  # Run SQL query
            else:  # PostgreSQL
                cursor.execute("SELECT version()")
            
            # .fetchone() gets one row of results
            # Returns a TUPLE like ('8.0.32',)
            version = cursor.fetchone()
            
            # INDEXING: [0] gets first element from tuple/list
            ver_str = version[0]
            
            print(f"✅ {db_type} Connected Successfully on {host}!")
            print(f"  Version: {ver_str}")
            
            # CLEANUP: Always close connections!
            cursor.close()
            conn.close()
            
            return True  # Exit function, return success
            
        except Exception as e:  # 'e' holds the error details
            print(f"  ⚠️ Failed to connect to {host}: {e}")
    
    # Only reaches here if ALL hosts failed
    print(f"❌ {db_type} Connection Failed.")
    return False

### 🎯 Key Concepts from `test_connection`:

| Concept | Syntax | Data Engineer Use Case |
|---------|--------|------------------------|
| f-string | `f"text {var}"` | Logging, SQL queries with variables |
| Dictionary | `{'key': 'value'}` | Config files, API responses, JSON |
| List | `[item1, item2]` | Storing multiple records, batch processing |
| try-except | `try: ... except:` | Handling DB errors, API failures |
| **kwargs | `**kwargs` | Flexible function parameters |
| for loop | `for x in items:` | Processing rows, files, tables |

### 2.2 Wrapper Functions: `test_mysql` and `test_postgres`

These are **simple functions** that call the main function with specific parameters.

**Why?** Makes the code cleaner and easier to call.

In [ ]:
def test_mysql():
    """Test MySQL connection with predefined settings."""
    # Calls test_connection with MySQL-specific parameters
    # Notice: we pass the FUNCTION itself (mysql.connector.connect)
    # not the result of calling it
    return test_connection(
        'MySQL',                    # db_type (positional argument)
        mysql.connector.connect,    # connector_func (the function to use)
        host="mysql",               # These become **kwargs
        user="devuser",
        password="devpassword",
        database="devdb"
    )

def test_postgres():
    """Test PostgreSQL connection with predefined settings."""
    return test_connection(
        'PostgreSQL',
        psycopg2.connect,
        host="postgres",
        user="devuser",
        password="devpassword",
        database="devdb"
    )

### 💡 Positional vs Keyword Arguments
```python
# Positional: order matters
test_connection('MySQL', mysql.connector.connect)

# Keyword: order doesn't matter, more readable
test_connection(db_type='MySQL', connector_func=mysql.connector.connect)

# Mixed: positional first, then keyword
test_connection('MySQL', mysql.connector.connect, host='localhost')
```

---
## 🚀 Part 3: The Main Entry Point

### What is `if __name__ == "__main__":`?

This is Python's way of saying: "Only run this code if this file is executed directly."

- If you run `python validate_dbs.py` → this code runs
- If you `import validate_dbs` in another file → this code does NOT run

**Why?** So you can reuse functions from this file without running the whole script.

In [ ]:
# This block only runs when script is executed directly
if __name__ == "__main__":
    print("Starting Database Validation...\n")  # \n = newline
    
    # NESTED TRY-EXCEPT: Check if required libraries are installed
    try:
        import mysql.connector
        import psycopg2
    except ImportError as e:  # ImportError = library not found
        print(f"❌ Missing required libraries: {e}")
        print("Please run: pip install mysql-connector-python psycopg2-binary")
        sys.exit(1)  # Exit with error code 1 (failure)

    # Call our test functions and store results
    mysql_success = test_mysql()      # Returns True or False
    print("-" * 40)                   # Print 40 dashes (string multiplication!)
    postgres_success = test_postgres()

    # FINAL SUMMARY
    print("\nSummary:")
    
    # BOOLEAN LOGIC: 'and' means BOTH must be True
    if mysql_success and postgres_success:
        print("✅ All Database Validations Passed!")
        sys.exit(0)  # Exit code 0 = success
    else:
        print("❌ Some Database Validations Failed!")
        sys.exit(1)  # Exit code 1 = failure

### 🎯 Exit Codes (Important for Data Engineering!)

| Code | Meaning | Use Case |
|------|---------|----------|
| 0 | Success | Pipeline continues |
| 1 | Failure | Pipeline stops, alerts triggered |

In data pipelines (Airflow, Jenkins, etc.), exit codes determine if the next step runs!

---
## 📝 Part 4: Quick Reference - Python Syntax Cheatsheet

### Data Types You'll Use Daily

In [ ]:
# STRING - text data
table_name = "users"
query = f"SELECT * FROM {table_name}"  # f-string interpolation

# INTEGER - whole numbers
row_count = 1000

# FLOAT - decimal numbers
avg_salary = 75000.50

# BOOLEAN - True/False
is_connected = True

# LIST - ordered, changeable collection
tables = ["users", "orders", "products"]
tables.append("inventory")  # Add item
first_table = tables[0]     # Access by index

# DICTIONARY - key-value pairs (like JSON!)
db_config = {
    "host": "localhost",
    "port": 5432,
    "database": "analytics"
}
host = db_config["host"]    # Access by key

# TUPLE - ordered, UNCHANGEABLE collection
row = ("John", 30, "Engineer")  # Often returned by database queries
name = row[0]

### Control Flow

In [ ]:
# IF-ELIF-ELSE
row_count = 100
if row_count == 0:
    print("No data")
elif row_count < 100:
    print("Small dataset")
else:
    print("Large dataset")

# FOR LOOP - iterate over items
tables = ["users", "orders"]
for table in tables:
    print(f"Processing {table}")

# WHILE LOOP - repeat until condition is false
retries = 3
while retries > 0:
    print(f"Attempt {4 - retries}")
    retries -= 1  # Same as: retries = retries - 1

---
## 🏆 Summary: What You Learned

1. **Imports** - Load external libraries (`import module`)
2. **Functions** - Reusable code blocks (`def func():`)
3. **Parameters** - Regular, `*args`, `**kwargs`
4. **Data Types** - str, int, float, bool, list, dict, tuple
5. **Control Flow** - if/elif/else, for, while
6. **Error Handling** - try/except (CRITICAL!)
7. **f-strings** - Modern string formatting
8. **Exit Codes** - 0=success, non-zero=failure
9. **`__name__ == "__main__"`** - Script entry point

### 🎯 Next Steps for Data Engineering:
- Learn **pandas** for data manipulation
- Practice **SQL** with Python
- Explore **SQLAlchemy** for ORM
- Study **error handling patterns** for ETL pipelines